In [44]:
!uv tree

learning-mcp v0.1.0
├── chromadb v1.1.1
│   ├── bcrypt v5.0.0
│   ├── build v1.3.0
│   │   ├── colorama v0.4.6
│   │   ├── packaging v25.0
│   │   └── pyproject-hooks v1.2.0
│   ├── grpcio v1.75.1
│   │   └── typing-extensions v4.15.0
│   ├── httpx v0.28.1
│   │   ├── anyio v4.11.0
│   │   │   ├── idna v3.10
│   │   │   ├── sniffio v1.3.1
│   │   │   └── typing-extensions v4.15.0
│   │   ├── certifi v2025.10.5
│   │   ├── httpcore v1.0.9
│   │   │   ├── certifi v2025.10.5
│   │   │   └── h11 v0.16.0
│   │   └── idna v3.10
│   ├── importlib-resources v6.5.2
│   ├── jsonschema v4.25.1
│   │   ├── attrs v25.4.0
│   │   ├── jsonschema-specifications v2025.9.1
│   │   │   └── referencing v0.36.2
│   │   │       ├── attrs v25.4.0
│   │   │       ├── rpds-py v0.27.1
│   │   │       └── typing-extensions v4.15.0
│   │   ├── referencing v0.36.2 (*)
│   │   ├── rpds-py v0.27.1
│   │   ├── fqdn v1.5.1 (extra: format-nongpl)
│   │   ├── idna v3.10 (extra: format-nongpl)
│   │   ├── isoduration v20

Resolved 203 packages in 2ms


# 0. Imports and Configuration Setup

In [45]:
# ----- IMPORT -----

import os
import shutil
import gc # garbage collection
import requests # get pdf from URL
from typing import List, Dict, Any, Optional # type hinting
from pathlib import Path # grt path directory


# LangChain imports
from langchain_chroma import Chroma # create vectordb and data ingestion
from langchain_ollama import OllamaEmbeddings # create embedding
from langchain_core.documents import Document # create Document object
from langchain.text_splitter import RecursiveCharacterTextSplitter # chunking data


# PDF processing
from PyPDF2 import PdfReader # read and conver pdf into text data


# FastMCP
from fastmcp import FastMCP # create FastMCP server

In [3]:
# current_dir = Path(__file__).parent # not working with .pynb
current_dir = Path.cwd()
current_dir


WindowsPath('c:/Users/user/Desktop/learning_mcp')

In [4]:
# ----- CONFIGURATION -----

CHROMA_PATH = os.path.join(current_dir, "chroma_db") # will be used for persistent storage
EMBED_MODEL = "nomic-embed-text"
OLLAMA_BASE_URL = "http://localhost:11434"
CHUNK_SIZE = 4096
CHUNK_OVERLAP = CHUNK_SIZE // 10 # 10% overlap
COLLECTION_NAME = "documents"


# 1. Initialization

In [5]:
# ----- MCP SERVER INITIALIZATION -----
mcp = FastMCP("langchain-chroma-vector-db")

In [6]:
# ----- OLLAMA EMBEDDING INITIALIZATION -----
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL
)

In [7]:
# ----- TEXT SPLITTER INITIALIZATION -----
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

In [8]:
# ----- CHROMA VECTOR STORE INITIALIZATION -----
vectorstore = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME
)

In [9]:
vectorstore.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [10]:
# # ----- CHROMA VECTOR STORE DELETION -----
# # Restart Jupyter kernel, then delete the chromadb directory

# CHROMA_PATH = "./chroma_db"
# if os.path.exists(CHROMA_PATH):
#     shutil.rmtree(CHROMA_PATH)
#     print(f"Deleted vectorstore at {CHROMA_PATH}")

# 2. Data Processing

We'll implement the following functions/methods that do all the heavy lifting for us:
1. `extract_text_from_pdf(pdf_path: str) -> str:` extract text from a pdf file
2. `process_single_pdf(pdf_path: str) -> int:` to extract text from pdf using `extract_text_from_pdf()`, create Document object, chunking, add chunks' metadata, then add to vectorstore
3. `download_pdf(url: str, download_dir: str = "./downloads") -> str:` to download pdf to `./download` folder and return the pdf file path

### Experiment

In [11]:
# pdf_path = "C:\Users\user\Desktop\learning_mcp\downloads\Statement_of_work_Exmp.pdf"
pdf_path = "./downloads\\Statement_of_work_Exmp.pdf"

In [12]:
reader = PdfReader(pdf_path)
reader

In [13]:
reader.pages

In [14]:
for page in reader.pages:
    print(page)
    page_text = page.extract_text()
    print(page_text)

{'/Type': '/Page', '/Parent': IndirectObject(2, 0, 3018612634112), '/Resources': {'/Font': {'/F1': IndirectObject(5, 0, 3018612634112), '/F2': IndirectObject(9, 0, 3018612634112), '/F3': IndirectObject(11, 0, 3018612634112)}, '/ExtGState': {'/GS7': IndirectObject(7, 0, 3018612634112), '/GS8': IndirectObject(8, 0, 3018612634112)}, '/ProcSet': ['/PDF', '/Text', '/ImageB', '/ImageC', '/ImageI']}, '/MediaBox': [0, 0, 612, 792], '/Contents': IndirectObject(4, 0, 3018612634112), '/Group': {'/Type': '/Group', '/S': '/Transparency', '/CS': '/DeviceRGB'}, '/Tabs': '/S', '/StructParents': 0}
c. Proposed STATEMENT OF WORK  (up to 2 pages)   
 
This proposal is to work to facilitate enrollment of producers in the IAMP program to receive 
incentives for implementing climate -smart practices on farms in XYZ county, Idaho, 
through contracts with the University of Idaho (U of I). We commit to working with 
producers at a Level of Engagement 3, from recruitment, through enrollment and 
implementation 

### 2.1 `extract_text_from_pdf()`

In [15]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from PDF using PyPDF"""
    try:
        # Extract text using PyPDF2
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        return text
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {e}")
        return ""

### Experiment

In [16]:
# pdf_path = "C:\Users\user\Desktop\learning_mcp\downloads\Statement_of_work_Exmp.pdf"
pdf_path = "./downloads\\Statement_of_work_Exmp.pdf"

In [17]:
Path(pdf_path).stem

'Statement_of_work_Exmp'

In [18]:
text = extract_text_from_pdf(pdf_path)
print(text)

c. Proposed STATEMENT OF WORK  (up to 2 pages)   
 
This proposal is to work to facilitate enrollment of producers in the IAMP program to receive 
incentives for implementing climate -smart practices on farms in XYZ county, Idaho, 
through contracts with the University of Idaho (U of I). We commit to working with 
producers at a Level of Engagement 3, from recruitment, through enrollment and 
implementation of contracts, through the entire term of each contract. Personnel from the 
XYZ Soil Conservation District will use our existing outreach platforms and contacts with 
producers and industry to promote the opportunity, answer questions of producers, and 
direct them to the IAMP online Phase 1 application if they have not already applied through 
that portal. We wil l review those applications and identify up to 5 producers to work with, 
assisting them with completion of the Phase 2 application. We will work with U of I 
personnel to produce the final contract based on the Phase 2 ap

In [19]:
doc = Document(
    page_content=text,
    metadata={
        "source": str(pdf_path),
        "filename": Path(pdf_path).name
    }
)
print(type(doc))
print(doc)

<class 'langchain_core.documents.base.Document'>
page_content='c. Proposed STATEMENT OF WORK  (up to 2 pages)   
 
This proposal is to work to facilitate enrollment of producers in the IAMP program to receive 
incentives for implementing climate -smart practices on farms in XYZ county, Idaho, 
through contracts with the University of Idaho (U of I). We commit to working with 
producers at a Level of Engagement 3, from recruitment, through enrollment and 
implementation of contracts, through the entire term of each contract. Personnel from the 
XYZ Soil Conservation District will use our existing outreach platforms and contacts with 
producers and industry to promote the opportunity, answer questions of producers, and 
direct them to the IAMP online Phase 1 application if they have not already applied through 
that portal. We wil l review those applications and identify up to 5 producers to work with, 
assisting them with completion of the Phase 2 application. We will work with U of I 


In [20]:
print([doc])

[Document(metadata={'source': './downloads\\Statement_of_work_Exmp.pdf', 'filename': 'Statement_of_work_Exmp.pdf'}, page_content='c. Proposed STATEMENT OF WORK  (up to 2 pages)   \n \nThis proposal is to work to facilitate enrollment of producers in the IAMP program to receive \nincentives for implementing climate -smart practices on farms in XYZ county, Idaho, \nthrough contracts with the University of Idaho (U of I). We commit to working with \nproducers at a Level of Engagement 3, from recruitment, through enrollment and \nimplementation of contracts, through the entire term of each contract. Personnel from the \nXYZ Soil Conservation District will use our existing outreach platforms and contacts with \nproducers and industry to promote the opportunity, answer questions of producers, and \ndirect them to the IAMP online Phase 1 application if they have not already applied through \nthat portal. We wil l review those applications and identify up to 5 producers to work with, \nassisti

In [21]:
chunks = text_splitter.split_documents([doc])
print(len(chunks))
chunks

2


[Document(metadata={'source': './downloads\\Statement_of_work_Exmp.pdf', 'filename': 'Statement_of_work_Exmp.pdf'}, page_content='c. Proposed STATEMENT OF WORK  (up to 2 pages)   \n \nThis proposal is to work to facilitate enrollment of producers in the IAMP program to receive \nincentives for implementing climate -smart practices on farms in XYZ county, Idaho, \nthrough contracts with the University of Idaho (U of I). We commit to working with \nproducers at a Level of Engagement 3, from recruitment, through enrollment and \nimplementation of contracts, through the entire term of each contract. Personnel from the \nXYZ Soil Conservation District will use our existing outreach platforms and contacts with \nproducers and industry to promote the opportunity, answer questions of producers, and \ndirect them to the IAMP online Phase 1 application if they have not already applied through \nthat portal. We wil l review those applications and identify up to 5 producers to work with, \nassisti

In [22]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i
    chunk.metadata["total_chunks"] = len(chunks)

In [23]:
chunks

[Document(metadata={'source': './downloads\\Statement_of_work_Exmp.pdf', 'filename': 'Statement_of_work_Exmp.pdf', 'chunk_index': 0, 'total_chunks': 2}, page_content='c. Proposed STATEMENT OF WORK  (up to 2 pages)   \n \nThis proposal is to work to facilitate enrollment of producers in the IAMP program to receive \nincentives for implementing climate -smart practices on farms in XYZ county, Idaho, \nthrough contracts with the University of Idaho (U of I). We commit to working with \nproducers at a Level of Engagement 3, from recruitment, through enrollment and \nimplementation of contracts, through the entire term of each contract. Personnel from the \nXYZ Soil Conservation District will use our existing outreach platforms and contacts with \nproducers and industry to promote the opportunity, answer questions of producers, and \ndirect them to the IAMP online Phase 1 application if they have not already applied through \nthat portal. We wil l review those applications and identify up t

In [24]:
ids = [f"{Path(pdf_path).stem}_chunk_{i}" for i in range(len(chunks))]
ids

['Statement_of_work_Exmp_chunk_0', 'Statement_of_work_Exmp_chunk_1']

In [25]:
vectorstore.add_documents(documents=chunks, ids=ids)

['Statement_of_work_Exmp_chunk_0', 'Statement_of_work_Exmp_chunk_1']

In [26]:
all_data = vectorstore.get()
all_data

{'ids': ['Statement_of_work_Exmp_chunk_0', 'Statement_of_work_Exmp_chunk_1'],
 'embeddings': None,
 'documents': ['c. Proposed STATEMENT OF WORK  (up to 2 pages)   \n \nThis proposal is to work to facilitate enrollment of producers in the IAMP program to receive \nincentives for implementing climate -smart practices on farms in XYZ county, Idaho, \nthrough contracts with the University of Idaho (U of I). We commit to working with \nproducers at a Level of Engagement 3, from recruitment, through enrollment and \nimplementation of contracts, through the entire term of each contract. Personnel from the \nXYZ Soil Conservation District will use our existing outreach platforms and contacts with \nproducers and industry to promote the opportunity, answer questions of producers, and \ndirect them to the IAMP online Phase 1 application if they have not already applied through \nthat portal. We wil l review those applications and identify up to 5 producers to work with, \nassisting them with co

In [29]:
# Get all IDs and delete them
if all_data['ids']:
    vectorstore.delete(ids=all_data['ids'])
    print(f"Deleted {len(all_data['ids'])} documents from vectorstore")


vectorstore.get()

Deleted 2 documents from vectorstore


{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [30]:
CHROMA_PATH

'c:\\Users\\user\\Desktop\\learning_mcp\\chroma_db'

In [31]:
# # Delete the vectorstore object
# del vectorstore

# # cleans up the memory and closes file handles
# gc.collect()

### 2.2 `process_single_pdf()`

In [59]:
def process_single_pdf(pdf_path: str) -> int:
    """Process a single PDF file and add to vectorstore"""
    text = extract_text_from_pdf(pdf_path)

    if not text:
        print(f"No text extracted from {pdf_path}")
        return 0

    # Create Document object
    doc = Document(
        page_content=text,
        metadata={
            "source": str(pdf_path),
            "filename": Path(pdf_path).name
        }
    )

    # Split into chunks using global text_splitter
    chunks = text_splitter.split_documents([doc])

    # Add chunk index to metadata
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_index"] = i
        chunk.metadata["total_chunks"] = len(chunks)

    # Add to vectorstore using global vectorstore
    ids = [f"{Path(pdf_path).stem}_chunk_{i}" for i in range(len(chunks))]
    vectorstore.add_documents(documents=chunks, ids=ids)

    return len(chunks)

### Experiment

In [32]:
download_dir: str = "./downloads"
os.makedirs(download_dir, exist_ok=True)

In [ ]:
url = "https://iamp.uidaho.edu/static/files/rfa_2025/attachments/Statement_of_work_Exmp.pdf?utm_source=chatgpt.com"
# url = "https://medium.com/data-science-collective/agentic-ai-implementing-long-term-memory-304be62063cc"
filename = Path(url.split("?")[0]).name

if not filename.endswith('.pdf'):
    filename = f"downloaded_{Path(url).stem}.pdf"

filename

'Statement_of_work_Exmp.pdf'

In [34]:
Path(url).stem

'Statement_of_work_Exmp.pdf?utm_source=chatgpt'

In [35]:
local_path = os.path.join(download_dir, filename)
local_path

'./downloads\\Statement_of_work_Exmp.pdf'

In [36]:
response = requests.get(url, stream=True)
response.raise_for_status()
response

<Response [200]>

In [37]:
response.iter_content(chunk_size=8192)

<generator object Response.iter_content.<locals>.generate at 0x000002BED40CD7E0>

In [38]:
with open(local_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

### 2.3 download_pdf()

In [60]:
def download_pdf(url: str, download_dir: str = "./downloads") -> str:
    """Download PDF from URL"""
    os.makedirs(download_dir, exist_ok=True)

    # Extract filename from URL
    filename = Path(url.split("?")[0]).name
    if not filename.endswith('.pdf'):
        filename = f"downloaded_{Path(url).stem}.pdf"

    local_path = os.path.join(download_dir, filename)

    # Download the file
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk) # write in chunks in case of big file size

    return local_path


# 3. MCP Tools Implementations

In this section, we'll implement the following tools:
1. `async def ingest_pdf(source: str) -> Dict[str, Any]:`
2. `async def retrieve(query: str, n: int = 5) -> List[Dict[str, Any]]:`
3. `async def db_info() -> Dict[str, Any]:`
4. `async def clear_db() -> Dict[str, Any]:`

### 1. `ingest_pdf()`

In [ ]:
@mcp.tool()
async def ingest_pdf(source: str) -> Dict[str, Any]:
    """
    Ingest PDF from folder, file path, or URL

    Args:
        source: Can be:
            - Folder path: processes all PDFs in folder
            - File path: processes single PDF
            - URL: downloads and processes PDF

    Returns:
        Status and number of chunks added
    """
    try:
        # Convert Path object to string
        source = str(source)

        total_chunks = 0
        processed_files = []

        # Handle URL
        if source.startswith(('http://', 'https://')):
            print(f"Downloading PDF from {source}")
            local_path = download_pdf(source)
            chunks = process_single_pdf(local_path)
            total_chunks += chunks
            processed_files.append(local_path)
            print(f"Processed {local_path}: {chunks} chunks")

        # Handle folder
        elif os.path.isdir(source):
            pdf_files = list(Path(source).glob("*.pdf"))
            print(f"Found {len(pdf_files)} PDF files in {source}")

            for pdf_file in pdf_files:
                print(f"Processing {pdf_file.name}...")
                chunks = process_single_pdf(str(pdf_file))
                total_chunks += chunks
                processed_files.append(str(pdf_file))
                print(f"Added {chunks} chunks from {pdf_file.name}")

        # Handle single file
        elif os.path.isfile(source) and source.endswith('.pdf'):
            print(f"Processing single file: {source}")
            chunks = process_single_pdf(source)
            total_chunks += chunks
            processed_files.append(source)
            print(f"Added {chunks} chunks")
            
        else:
            return {
                "status": "error",
                "message": f"Invalid source: {source}. Must be a PDF file, or an URL."
            }

        return {
            "status": "success",
            "chunk_added": total_chunks,
            "files_processed": len(processed_files),
            "files": processed_files
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }



### Experiment

In [51]:
type(ingest_pdf)

fastmcp.tools.tool.FunctionTool

In [ ]:
# # Get all IDs and delete them
# if vectorstore.get()['ids']:
#     vectorstore.delete(ids=all_data['ids'])
#     print(f"Deleted {len(all_data['ids'])} documents from vectorstore")


vectorstore.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [ ]:
# Error left on purpose
url = "https://github.com/py-pdf/sample-files/blob/main/004-pdflatex-4-pages/pdflatex-4-pages.pdf"
folder_path = "C:\Users\user\Desktop\learning_mcp\downloads"
file_path = "C:\Users\user\Desktop\learning_mcp\downloads\pdflatex-4-pages.pdf"

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (1061439956.py, line 2)

In [64]:
url = "https://github.com/py-pdf/sample-files/raw/main/004-pdflatex-4-pages/pdflatex-4-pages.pdf"
# folder_path = Path(r"C:\Users\user\Desktop\learning_mcp\downloads") # raw string (prefix with "r")
# folder_path = Path("C:/Users/user/Desktop/learning_mcp/downloads") # forward slashes
# folder_path = Path("C:\\Users\\user\\Desktop\\learning_mcp\\downloads") # escape the backslashes
folder_path = Path("downloads") # relative path
file_path = folder_path / "pdflatex-4-pages.pdf"

In [70]:
# Test that all functions are accessible
print("extract_text_from_pdf:", callable(extract_text_from_pdf))
print("process_single_pdf:", callable(process_single_pdf))
print("download_pdf:", callable(download_pdf))
print("ingest_pdf:", ingest_pdf)

extract_text_from_pdf: True
process_single_pdf: True
download_pdf: True
ingest_pdf: <function ingest_pdf at 0x000002BED3743740>


In [66]:
# comment out mcp.tool() decorator first
ingestion_result = await ingest_pdf(url)
print(ingestion_result)

Processed ./downloads\pdflatex-4-pages.pdf: 4 chunks
{'status': 'success', 'chunk_added': 4, 'files_processed': 1, 'files': ['./downloads\\pdflatex-4-pages.pdf']}


In [67]:
vectorstore.get()

{'ids': ['pdflatex-4-pages_chunk_0',
  'pdflatex-4-pages_chunk_1',
  'pdflatex-4-pages_chunk_2',
  'pdflatex-4-pages_chunk_3'],
 'embeddings': None,
 'documents': ['Hello, here is some text without a meaning. This text should show what a printed text\nwill look like at this place. If you read this text, you will get no information. Really?\nIs there no information? Is there a difference between this text and some nonsense like\n“Huardest gefburn”? Kjift – not at all! A blind text like this gives you information\nabout the selected font, how the letters are written and an impression of the look.\nThis text should contain all letters of the alphabet and it should be written in of the\noriginal language. There is no need for special content, but the length of words should\nmatch the language. Hello, here is some text without a meaning. This text should show\nwhat a printed text will look like at this place. If you read this text, you will get no\ninformation. Really? Is there no informati

In [71]:
# comment out mcp.tool() decorator first
ingestion_result = await ingest_pdf(folder_path)
print(ingestion_result)

Found 2 PDF files in downloads
Processing pdflatex-4-pages.pdf...
Added 4 chunks from pdflatex-4-pages.pdf
Processing Statement_of_work_Exmp.pdf...
Added 2 chunks from Statement_of_work_Exmp.pdf
{'status': 'success', 'chunk_added': 6, 'files_processed': 2, 'files': ['downloads\\pdflatex-4-pages.pdf', 'downloads\\Statement_of_work_Exmp.pdf']}


In [78]:
len(vectorstore.get()['ids']), vectorstore.get()

(6,
 {'ids': ['pdflatex-4-pages_chunk_0',
   'pdflatex-4-pages_chunk_1',
   'pdflatex-4-pages_chunk_2',
   'pdflatex-4-pages_chunk_3',
   'Statement_of_work_Exmp_chunk_0',
   'Statement_of_work_Exmp_chunk_1'],
  'embeddings': None,
  'documents': ['Hello, here is some text without a meaning. This text should show what a printed text\nwill look like at this place. If you read this text, you will get no information. Really?\nIs there no information? Is there a difference between this text and some nonsense like\n“Huardest gefburn”? Kjift – not at all! A blind text like this gives you information\nabout the selected font, how the letters are written and an impression of the look.\nThis text should contain all letters of the alphabet and it should be written in of the\noriginal language. There is no need for special content, but the length of words should\nmatch the language. Hello, here is some text without a meaning. This text should show\nwhat a printed text will look like at this place

In [79]:
# comment out mcp.tool() decorator first
ingestion_result = await ingest_pdf(file_path)
print(ingestion_result)

Processing single file: downloads\pdflatex-4-pages.pdf
Added 4 chunks
{'status': 'success', 'chunk_added': 4, 'files_processed': 1, 'files': ['downloads\\pdflatex-4-pages.pdf']}


In [ ]:
len(vectorstore.get()['ids']), vectorstore.get()
# note: chunks are overwrited because we gave them the same name

(6,
 {'ids': ['pdflatex-4-pages_chunk_0',
   'pdflatex-4-pages_chunk_1',
   'pdflatex-4-pages_chunk_2',
   'pdflatex-4-pages_chunk_3',
   'Statement_of_work_Exmp_chunk_0',
   'Statement_of_work_Exmp_chunk_1'],
  'embeddings': None,
  'documents': ['Hello, here is some text without a meaning. This text should show what a printed text\nwill look like at this place. If you read this text, you will get no information. Really?\nIs there no information? Is there a difference between this text and some nonsense like\n“Huardest gefburn”? Kjift – not at all! A blind text like this gives you information\nabout the selected font, how the letters are written and an impression of the look.\nThis text should contain all letters of the alphabet and it should be written in of the\noriginal language. There is no need for special content, but the length of words should\nmatch the language. Hello, here is some text without a meaning. This text should show\nwhat a printed text will look like at this place

In [82]:
# Get all IDs and delete them
all_chunks_ids = vectorstore.get()['ids']
if all_chunks_ids:
    vectorstore.delete(ids=all_chunks_ids)
    print(f"Deleted {len(all_chunks_ids)} documents from vectorstore")


vectorstore.get()

Deleted 6 documents from vectorstore


{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

`DON'T FORGET TO UNCOMMENT @mcp.tool() DECORATOR`

### 2. `retrieve()`

In [83]:
@mcp.tool()
async def retrieve(query: str, n: int = 5) -> List[Dict[str, Any]]:
    """
    Retrieve top N chunks for given query

    Args:
        query: Search query
        n: Number of chunks to retrieve

    Returns:
        Top N matching chunks with scores
    """
    try:
        # Use global vectorstore directly
        results = vectorstore.similarity_search_with_score(query, k=n)

        # Format results
        chunks = []
        for doc, score in results:
            chunks.append({
                "text": doc.page_content,
                "metadata": doc.metadata,
                "similarity_score": float(1 - score), # Convert distance
                "distance": float(score)
            })
        
        return chunks
    
    except Exception as e:
        return [{"error": str(e)}]

### 3. `db_info()`

In [90]:
@mcp.tool()
async def db_info() -> Dict[str, Any]:
    """
    Get ChromaDB and collection information

    Returns:
        Database and collection statistics
    """
    try:
        # Use global vectorstore directly
        collection = vectorstore._collection

        # Get count (i.e. total number of doc chunks)
        count = collection.count()

        # Get sample documents to extract unique sources
        sample_size = min(100, count) if count > 0 else 0
        sources = set()

        if sample_size > 0:
            sample = collection.get(
                limit=sample_size,
                include=["metadatas"]
            )

            if sample["metadatas"]:
                for metadata in sample["metadatas"]:
                    if metadata and  "source" in metadata:
                        sources.add(metadata["source"])

        return {
            "database_path": CHROMA_PATH,
            "collection_name": COLLECTION_NAME,
            "embedding_model": EMBED_MODEL,
            "ollama_base_url": OLLAMA_BASE_URL,
            "total_chunks": count,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "unique_source": list(sources),
            "num_sources": len(sources)
        }

    except Exception as e:
        return {
            "error": str(e)
        }

[10/13/25 15:02:18] WARNING  Tool already exists: db_info                                       ]8;id=872734;file://c:\Users\user\Desktop\learning_mcp\.venv\Lib\site-packages\fastmcp\tools\tool_manager.py\tool_manager.py]8;;\:]8;id=340356;file://c:\Users\user\Desktop\learning_mcp\.venv\Lib\site-packages\fastmcp\tools\tool_manager.py#171\171]8;;\

### 4. `clear_db()`

In [91]:
@mcp.tool()
async def clear_db() -> Dict[str, Any]:
    """
    Clear all data from database

    Returns:
        Confimation message
    """
    try:
        global vectorstore
        # using global since we reassign it later inside the function

        # Delete the collection
        vectorstore.delete_collection()

        # Recreate empty vectorsore with same configuration (so that other tool can use it after that)
        vectorstore = Chroma(
            persist_directory=CHROMA_PATH,
            embedding_function=embeddings,
            collection_name=COLLECTION_NAME
        )

        return {
            "status": "success",
            "message": "Database cleared and reset"
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

[10/13/25 15:02:29] WARNING  Tool already exists: clear_db                                      ]8;id=152522;file://c:\Users\user\Desktop\learning_mcp\.venv\Lib\site-packages\fastmcp\tools\tool_manager.py\tool_manager.py]8;;\:]8;id=61891;file://c:\Users\user\Desktop\learning_mcp\.venv\Lib\site-packages\fastmcp\tools\tool_manager.py#171\171]8;;\

>  what are the differences between `vectorstore.delete()` and `vectorstore.delete_collection()`?

⛬  The key differences between these two methods:

`vectorstore.delete()`

•  Deletes specific documents/vectors from within a collection

•  Takes parameters like ids or filter to specify which documents to remove

•  The collection structure remains intact

•  Use when you want to remove certain documents but keep the collection

`vectorstore.delete_collection()`

•  Deletes the entire collection itself

•  Removes all documents AND the collection structure

•  No parameters needed (operates on the whole collection)

•  Use when you want to completely remove the collection and start fresh

Think of it like files vs folders: delete() removes specific files from a folder, while
delete_collection() removes the entire folder.

>  what are differences between documents/vectors and collection?

Collection

•  A named container/namespace that holds related documents   

•  Like a "database table" or "folder"

•  Has its own configuration (embedding function, metadata, etc.)

•  Example: "legal_documents", "medical_papers", "user_manuals"  

Documents/Vectors

•  Individual pieces of content stored inside a collection      

•  Each document has:

    •  Text content: The actual text/chunk

    •  Vector embedding: Numerical representation of the text

    •  Metadata: Additional info (source, page number, date, etc.)

    •  ID: Unique identifier

•  Example: One document could be a paragraph from page 3 of a PDF

Relationship

    Collection "research_papers"
    ├── Document 1: {text: "Introduction...", vector: [0.1, 0.2, ...], metadata: {page: 1}}
    ├── Document 2: {text: "Methods...", vector: [0.3, 0.4, ...], metadata: {page: 2}}
    └── Document 3: {text: "Results...", vector: [0.5, 0.6, ...], metadata: {page: 3}}

So a collection is the container, and documents/vectors are the individual items stored
within it.